# Apache Spark Practice Notebook
##### ลองฝึก PySpark ตัวอย่างข้อมูล Food.com Recipes and Reviews Dataset

Dataset:
`irkaal/foodcom-recipes-and-reviews`

สิ่งที่ได้ฝึก
- โหลด dataset
- SparkSession
- อ่าน CSV
- ดูข้อมูลด้วย `show`, `head`, `first`, `take`
- เช็ก schema / data type / columns
- เลือก column
- แปลงชนิดข้อมูล
- clean missing values และ duplicates
- ใช้ `select`, `when`, `like`, `contains`, `startswith`, `endswith`, `substr`, `between`
- ใช้ `groupBy`, `agg`, `filter`, `sort`, `orderBy`
- join DataFrame
- ใช้ Spark SQL
- save output เป็น Parquet / CSV / JSON


## 0. ติดตั้งและดาวน์โหลด Dataset


In [111]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "irkaal/foodcom-recipes-and-reviews"
)

print(dataset_path)

Using Colab cache for faster access to the 'foodcom-recipes-and-reviews' dataset.
/kaggle/input/foodcom-recipes-and-reviews



- `import kagglehub`  
  เรียกใช้ package สำหรับดาวน์โหลด dataset จาก Kaggle

- `kagglehub.dataset_download(...)`  
  ดาวน์โหลด dataset ตามชื่อ dataset slug

- `"irkaal/foodcom-recipes-and-reviews"`  
  คือชื่อ dataset ที่ต้องการโหลด

- `dataset_path`  
  คือ path ของ folder ที่เก็บไฟล์หลัง download เสร็จ

In [112]:
#  ===== วิธีโหลดไฟล์ใน Spark =====

#  1) RDD (low-level)
    # from pyspark import SparkContext
    # sc = SparkContext.getOrCreate()

    # rdd_txt = sc.textFile("file.txt")
    # rdd_csv = sc.textFile("file.csv")

# 2) DataFrame
  # JSON
  # df_json = spark.read.json("file.json")

  # Parquet
  # df_parquet = spark.read.load("file.parquet")

  # TXT
  # df_txt = spark.read.text("file.txt")

  # CSV
  # df_csv = spark.read.csv("file.csv", header=True, inferSchema=True)


In [113]:
import os

os.listdir(dataset_path)

['recipes.parquet', 'reviews.parquet', 'reviews.csv', 'recipes.csv']

ใช้ดูว่าใน dataset folder มีไฟล์อะไรบ้าง  


## 1. SparkSession

`SparkSession`
ใช้สำหรับอ่านข้อมูล, สร้าง DataFrame, ใช้ SQL และ save output

In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("FoodCom Spark Practice TH")
    .getOrCreate()
)

- `SparkSession.builder`  
  เริ่มสร้าง Spark application

- `.appName(...)`  
  ตั้งชื่อ application เพื่อดูใน log หรือ Spark UI

- `.getOrCreate()`  
  ถ้ามี SparkSession อยู่แล้วจะใช้ของเดิม ถ้ายังไม่มีจะสร้างใหม่

### (Scala) SparkSession

In [114]:
# Scala

'''
import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder()
  .appName("FoodCom Spark Practice")
  .getOrCreate()
'''

'\nimport org.apache.spark.sql.SparkSession\n\nval spark = SparkSession.builder()\n  .appName("FoodCom Spark Practice")\n  .getOrCreate()\n'

## 2. อ่าน CSV แบบปกติ

เพื่อลด error จากการเดา schema ผิด ให้อ่าน CSV แบบ basic ก่อน โดยยังไม่ใช้ `inferSchema=True`

จากนั้นค่อย cast column ที่จำเป็นเอง

In [115]:
recipes_path = f"{dataset_path}/recipes.csv"
reviews_path = f"{dataset_path}/reviews.csv"

recipes_df = spark.read.csv(recipes_path, header=True, multiLine=True, escape='"')
reviews_df = spark.read.csv(reviews_path, header=True, multiLine=True, escape='"')

- `spark.read.csv(...)`  
  อ่านไฟล์ CSV เป็น Spark DataFrame

- `header=True`  
  ใช้แถวแรกเป็นชื่อ column

- ไม่ใช้ `inferSchema=True`  
  ทำให้ Spark อ่านข้อมูลเป็น string ก่อน วิธีนี้ basic และลดโอกาส error จากการเดาชนิดข้อมูลผิด

### (Scala) อ่าน CSV

In [116]:
# Scala note เท่านั้น

'''
val recipesDF = spark.read
  .option("header", "true")
  .csv("recipes.csv")
'''

'\nval recipesDF = spark.read\n  .option("header", "true")\n  .csv("recipes.csv")\n'

## 3. Inspect Data ดูข้อมูลเบื้องต้น



In [117]:
recipes_df.show()

+--------+--------------------+--------+--------------------+--------+--------+---------+--------------------+--------------------+--------------------+-----------------+--------------------+--------------------------+---------------------+----------------+-----------+--------+----------+-------------------+------------------+-------------+-------------------+------------+------------+--------------+--------------+------------+--------------------+
|RecipeId|                Name|AuthorId|          AuthorName|CookTime|PrepTime|TotalTime|       DatePublished|         Description|              Images|   RecipeCategory|            Keywords|RecipeIngredientQuantities|RecipeIngredientParts|AggregatedRating|ReviewCount|Calories|FatContent|SaturatedFatContent|CholesterolContent|SodiumContent|CarbohydrateContent|FiberContent|SugarContent|ProteinContent|RecipeServings| RecipeYield|  RecipeInstructions|
+--------+--------------------+--------+--------------------+--------+--------+---------+-----

In [118]:

recipes_df.show(5)

+--------+--------------------+--------+--------------+--------+--------+---------+--------------------+--------------------+--------------------+---------------+--------------------+--------------------------+---------------------+----------------+-----------+--------+----------+-------------------+------------------+-------------+-------------------+------------+------------+--------------+--------------+-----------+--------------------+
|RecipeId|                Name|AuthorId|    AuthorName|CookTime|PrepTime|TotalTime|       DatePublished|         Description|              Images| RecipeCategory|            Keywords|RecipeIngredientQuantities|RecipeIngredientParts|AggregatedRating|ReviewCount|Calories|FatContent|SaturatedFatContent|CholesterolContent|SodiumContent|CarbohydrateContent|FiberContent|SugarContent|ProteinContent|RecipeServings|RecipeYield|  RecipeInstructions|
+--------+--------------------+--------+--------------+--------+--------+---------+--------------------+--------

- `show(5)`  
  แสดงข้อมูล 5 แถวแรกในรูปแบบตาราง

- `show()` เป็น Action  
  แปลว่า Spark จะเริ่มอ่านข้อมูลจริงตอนเจอคำสั่งนี้

In [119]:
recipes_df.printSchema()

root
 |-- RecipeId: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- AuthorId: string (nullable = true)
 |-- AuthorName: string (nullable = true)
 |-- CookTime: string (nullable = true)
 |-- PrepTime: string (nullable = true)
 |-- TotalTime: string (nullable = true)
 |-- DatePublished: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Images: string (nullable = true)
 |-- RecipeCategory: string (nullable = true)
 |-- Keywords: string (nullable = true)
 |-- RecipeIngredientQuantities: string (nullable = true)
 |-- RecipeIngredientParts: string (nullable = true)
 |-- AggregatedRating: string (nullable = true)
 |-- ReviewCount: string (nullable = true)
 |-- Calories: string (nullable = true)
 |-- FatContent: string (nullable = true)
 |-- SaturatedFatContent: string (nullable = true)
 |-- CholesterolContent: string (nullable = true)
 |-- SodiumContent: string (nullable = true)
 |-- CarbohydrateContent: string (nullable = true)
 |-- FiberContent: stri

- `printSchema()`  
  แสดงโครงสร้างของ DataFrame เช่น ชื่อ column และ data type

In [120]:
recipes_df.dtypes

[('RecipeId', 'string'),
 ('Name', 'string'),
 ('AuthorId', 'string'),
 ('AuthorName', 'string'),
 ('CookTime', 'string'),
 ('PrepTime', 'string'),
 ('TotalTime', 'string'),
 ('DatePublished', 'string'),
 ('Description', 'string'),
 ('Images', 'string'),
 ('RecipeCategory', 'string'),
 ('Keywords', 'string'),
 ('RecipeIngredientQuantities', 'string'),
 ('RecipeIngredientParts', 'string'),
 ('AggregatedRating', 'string'),
 ('ReviewCount', 'string'),
 ('Calories', 'string'),
 ('FatContent', 'string'),
 ('SaturatedFatContent', 'string'),
 ('CholesterolContent', 'string'),
 ('SodiumContent', 'string'),
 ('CarbohydrateContent', 'string'),
 ('FiberContent', 'string'),
 ('SugarContent', 'string'),
 ('ProteinContent', 'string'),
 ('RecipeServings', 'string'),
 ('RecipeYield', 'string'),
 ('RecipeInstructions', 'string')]



- `dtypes`  
  แสดงชื่อ column พร้อม data type เป็น list

ใช้เช็กเร็ว ๆ ว่าแต่ละ column ถูกอ่านเป็นชนิดอะไร

In [121]:
recipes_df.columns

['RecipeId',
 'Name',
 'AuthorId',
 'AuthorName',
 'CookTime',
 'PrepTime',
 'TotalTime',
 'DatePublished',
 'Description',
 'Images',
 'RecipeCategory',
 'Keywords',
 'RecipeIngredientQuantities',
 'RecipeIngredientParts',
 'AggregatedRating',
 'ReviewCount',
 'Calories',
 'FatContent',
 'SaturatedFatContent',
 'CholesterolContent',
 'SodiumContent',
 'CarbohydrateContent',
 'FiberContent',
 'SugarContent',
 'ProteinContent',
 'RecipeServings',
 'RecipeYield',
 'RecipeInstructions']

- `columns`  
  แสดงชื่อ column ทั้งหมดใน DataFrame เป็น list

In [62]:
recipes_df.count()

522517

522517

- `count()` นับจำนวน row ทั้งหมด ถ้าข้อมูลใหญ่มากอาจใช้เวลานาน

In [63]:
reviews_df.show(5)
reviews_df.printSchema()
reviews_df.count()

+--------+--------+--------+----------------+------+--------------------+--------------------+--------------------+
|ReviewId|RecipeId|AuthorId|      AuthorName|Rating|              Review|       DateSubmitted|        DateModified|
+--------+--------+--------+----------------+------+--------------------+--------------------+--------------------+
|       2|     992|    2008|       gayg msft|     5|better than any y...|2000-01-25T21:44:00Z|2000-01-25T21:44:00Z|
|       7|    4384|    1634|   Bill Hilbrich|     4|I cut back on the...|2001-10-17T16:49:59Z|2001-10-17T16:49:59Z|
|       9|    4523|    2046|Gay Gilmore ckpt|     2|i think i did som...|2000-02-25T09:00:00Z|2000-02-25T09:00:00Z|
|      13|    7435|    1773|   Malarkey Test|     5|easily the best i...|2000-03-13T21:15:00Z|2000-03-13T21:15:00Z|
|      14|      44|    2085|      Tony Small|     5|  An excellent dish.|2000-03-28T12:51:00Z|2000-03-28T12:51:00Z|
+--------+--------+--------+----------------+------+--------------------

1401982

+--------+--------+--------+----------------+------+--------------------+--------------------+--------------------+
|ReviewId|RecipeId|AuthorId|      AuthorName|Rating|              Review|       DateSubmitted|        DateModified|
+--------+--------+--------+----------------+------+--------------------+--------------------+--------------------+
|       2|     992|    2008|       gayg msft|     5|better than any y...|2000-01-25T21:44:00Z|2000-01-25T21:44:00Z|
|       7|    4384|    1634|   Bill Hilbrich|     4|I cut back on the...|2001-10-17T16:49:59Z|2001-10-17T16:49:59Z|
|       9|    4523|    2046|Gay Gilmore ckpt|     2|i think i did som...|2000-02-25T09:00:00Z|2000-02-25T09:00:00Z|
|      13|    7435|    1773|   Malarkey Test|     5|easily the best i...|2000-03-13T21:15:00Z|2000-03-13T21:15:00Z|
|      14|      44|    2085|      Tony Small|     5|  An excellent dish.|2000-03-28T12:51:00Z|2000-03-28T12:51:00Z|
+--------+--------+--------+----------------+------+--------------------

1401982

## 4. ดูข้อมูลหลายแบบ: show, head, first, take

In [64]:
recipes_df.show(3)

+--------+--------------------+--------+--------------+--------+--------+---------+--------------------+--------------------+--------------------+---------------+--------------------+--------------------------+---------------------+----------------+-----------+--------+----------+-------------------+------------------+-------------+-------------------+------------+------------+--------------+--------------+-----------+--------------------+
|RecipeId|                Name|AuthorId|    AuthorName|CookTime|PrepTime|TotalTime|       DatePublished|         Description|              Images| RecipeCategory|            Keywords|RecipeIngredientQuantities|RecipeIngredientParts|AggregatedRating|ReviewCount|Calories|FatContent|SaturatedFatContent|CholesterolContent|SodiumContent|CarbohydrateContent|FiberContent|SugarContent|ProteinContent|RecipeServings|RecipeYield|  RecipeInstructions|
+--------+--------------------+--------+--------------+--------+--------+---------+--------------------+--------

- `show(3)`  แสดง 3 แถวแรกเป็นตาราง

In [65]:
recipes_df.head(3)

[Row(RecipeId='38', Name='Low-Fat Berry Blue Frozen Dessert', AuthorId='1533', AuthorName='Dancer', CookTime='PT24H', PrepTime='PT45M', TotalTime='PT24H45M', DatePublished='1999-08-09T21:46:00Z', Description='Make and share this Low-Fat Berry Blue Frozen Dessert recipe from Food.com.', Images='c("https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/YUeirxMLQaeE1h3v3qnM_229%20berry%20blue%20frzn%20dess.jpg", "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/AFPDDHATWzQ0b1CDpDAT_255%20berry%20blue%20frzn%20dess.jpg", "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/UYgf9nwMT2SGGJCuzILO_228%20berry%20blue%20frzn%20dess.jpg", "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/PeBMJN2TGSaYks2759BA_20140722_202142.jpg", \n"https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img

[Row(RecipeId='38', Name='Low-Fat Berry Blue Frozen Dessert', AuthorId='1533', AuthorName='Dancer', CookTime='PT24H', PrepTime='PT45M', TotalTime='PT24H45M', DatePublished='1999-08-09T21:46:00Z', Description='Make and share this Low-Fat Berry Blue Frozen Dessert recipe from Food.com.', Images='c("https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/YUeirxMLQaeE1h3v3qnM_229%20berry%20blue%20frzn%20dess.jpg", "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/AFPDDHATWzQ0b1CDpDAT_255%20berry%20blue%20frzn%20dess.jpg", "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/UYgf9nwMT2SGGJCuzILO_228%20berry%20blue%20frzn%20dess.jpg", "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/PeBMJN2TGSaYks2759BA_20140722_202142.jpg", \n"https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img

- `head(3)`  ดึง 3 แถวแรกออกมาเป็น list ของ Row

ใช้กับข้อมูลจำนวนน้อยเท่านั้น เพราะข้อมูลถูกดึงกลับมาที่ driver

In [66]:
recipes_df.first()

Row(RecipeId='38', Name='Low-Fat Berry Blue Frozen Dessert', AuthorId='1533', AuthorName='Dancer', CookTime='PT24H', PrepTime='PT45M', TotalTime='PT24H45M', DatePublished='1999-08-09T21:46:00Z', Description='Make and share this Low-Fat Berry Blue Frozen Dessert recipe from Food.com.', Images='c("https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/YUeirxMLQaeE1h3v3qnM_229%20berry%20blue%20frzn%20dess.jpg", "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/AFPDDHATWzQ0b1CDpDAT_255%20berry%20blue%20frzn%20dess.jpg", "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/UYgf9nwMT2SGGJCuzILO_228%20berry%20blue%20frzn%20dess.jpg", "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/PeBMJN2TGSaYks2759BA_20140722_202142.jpg", \n"https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/

Row(RecipeId='38', Name='Low-Fat Berry Blue Frozen Dessert', AuthorId='1533', AuthorName='Dancer', CookTime='PT24H', PrepTime='PT45M', TotalTime='PT24H45M', DatePublished='1999-08-09T21:46:00Z', Description='Make and share this Low-Fat Berry Blue Frozen Dessert recipe from Food.com.', Images='c("https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/YUeirxMLQaeE1h3v3qnM_229%20berry%20blue%20frzn%20dess.jpg", "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/AFPDDHATWzQ0b1CDpDAT_255%20berry%20blue%20frzn%20dess.jpg", "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/UYgf9nwMT2SGGJCuzILO_228%20berry%20blue%20frzn%20dess.jpg", "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/PeBMJN2TGSaYks2759BA_20140722_202142.jpg", \n"https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/

- `first()`  
  ดึงแถวแรกของ DataFrame

In [67]:
recipes_df.take(2)

[Row(RecipeId='38', Name='Low-Fat Berry Blue Frozen Dessert', AuthorId='1533', AuthorName='Dancer', CookTime='PT24H', PrepTime='PT45M', TotalTime='PT24H45M', DatePublished='1999-08-09T21:46:00Z', Description='Make and share this Low-Fat Berry Blue Frozen Dessert recipe from Food.com.', Images='c("https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/YUeirxMLQaeE1h3v3qnM_229%20berry%20blue%20frzn%20dess.jpg", "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/AFPDDHATWzQ0b1CDpDAT_255%20berry%20blue%20frzn%20dess.jpg", "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/UYgf9nwMT2SGGJCuzILO_228%20berry%20blue%20frzn%20dess.jpg", "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/PeBMJN2TGSaYks2759BA_20140722_202142.jpg", \n"https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img

[Row(RecipeId='38', Name='Low-Fat Berry Blue Frozen Dessert', AuthorId='1533', AuthorName='Dancer', CookTime='PT24H', PrepTime='PT45M', TotalTime='PT24H45M', DatePublished='1999-08-09T21:46:00Z', Description='Make and share this Low-Fat Berry Blue Frozen Dessert recipe from Food.com.', Images='c("https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/YUeirxMLQaeE1h3v3qnM_229%20berry%20blue%20frzn%20dess.jpg", "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/AFPDDHATWzQ0b1CDpDAT_255%20berry%20blue%20frzn%20dess.jpg", "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/UYgf9nwMT2SGGJCuzILO_228%20berry%20blue%20frzn%20dess.jpg", "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/PeBMJN2TGSaYks2759BA_20140722_202142.jpg", \n"https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img

- `take(2)`  
  ดึง 2 แถวแรกออกมาเป็น list ของ Row คล้าย `head()`

## 5. เลือกเฉพาะ Column ที่จะใช้



In [68]:
recipes_small = recipes_df.select(
    "RecipeId",
    "Name",
    "RecipeCategory",
    "CookTime",
    "PrepTime",
    "TotalTime",
    "RecipeServings",
    "RecipeIngredientParts",
    "Calories"
)

reviews_small = reviews_df.select(
    "RecipeId",
    "AuthorId",
    "Rating",
    "Review",
    "DateSubmitted"
)

- `select(...)`  
  เลือกเฉพาะ column ที่ต้องการใช้

- `recipes_small`  
  DataFrame ของข้อมูลสูตรอาหารแบบลด column แล้ว

- `reviews_small`  
  DataFrame ของข้อมูล review แบบลด column แล้ว



> Spark DataFrame เป็น immutable
คือคำสั่งพวกนี้จะสร้าง DataFrame ใหม่ไม่ได้ไปแก้ของเดิม

## 6. แปลงชนิดข้อมูล + Clean ข้อมูล

In [69]:
from pyspark.sql import functions as F

recipes_clean = (
    recipes_small
    .withColumn("RecipeId", F.col("RecipeId").cast("int"))
    .withColumn("Calories", F.expr("try_cast(Calories as double)"))
    .dropna(subset=["RecipeId"])
    .dropDuplicates(["RecipeId"])
)

reviews_clean = (
    reviews_small
    .withColumn("RecipeId", F.col("RecipeId").cast("int"))
    .withColumn("Rating", F.col("Rating").cast("double"))
    .dropna(subset=["RecipeId", "Rating"])
)

- `from pyspark.sql import functions as F`  
  import function ที่ใช้บ่อยใน Spark เช่น `col`, `avg`, `count`, `when`

- `withColumn("RecipeId", F.col("RecipeId").cast("int"))`  
  แปลง `RecipeId` เป็น integer

- `withColumn("Calories", F.col("Calories").cast("double"))`  
  แปลง `Calories` เป็นเลขทศนิยม

- `withColumn("Rating", F.col("Rating").cast("double"))`  
  แปลง `Rating` เป็นเลขทศนิยม

- `dropna(subset=[...])`  
  ลบ row ที่ column สำคัญเป็น null

- `dropDuplicates(["RecipeId"])`  
  ลบที่มี `RecipeId` ซ้ำ

In [70]:
recipes_clean.printSchema()
reviews_clean.printSchema()

root
 |-- RecipeId: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- RecipeCategory: string (nullable = true)
 |-- CookTime: string (nullable = true)
 |-- PrepTime: string (nullable = true)
 |-- TotalTime: string (nullable = true)
 |-- RecipeServings: string (nullable = true)
 |-- RecipeIngredientParts: string (nullable = true)
 |-- Calories: double (nullable = true)

root
 |-- RecipeId: integer (nullable = true)
 |-- AuthorId: string (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Review: string (nullable = true)
 |-- DateSubmitted: string (nullable = true)

root
 |-- RecipeId: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- RecipeCategory: string (nullable = true)
 |-- CookTime: string (nullable = true)
 |-- PrepTime: string (nullable = true)
 |-- TotalTime: string (nullable = true)
 |-- RecipeServings: string (nullable = true)
 |-- RecipeIngredientParts: string (nullable = true)
 |-- Calories: double (nullable = true)

root
 |-- Reci

เช็ก schema อีกครั้งว่า column ที่ต้องใช้คำนวณถูกแปลงเป็นตัวเลขแล้วหรือยัง

In [71]:
from pyspark.sql import functions as F

recipes_clean = (
    recipes_small
    .withColumn("RecipeId", F.expr("try_cast(RecipeId as int)"))
    .withColumn("Calories", F.regexp_replace("Calories", "[^0-9.]", ""))
    .withColumn("Calories", F.expr("try_cast(Calories as double)"))
    .dropna(subset=["RecipeId"])
    .dropDuplicates(["RecipeId"])
)

reviews_clean = (
    reviews_small
    .withColumn("RecipeId", F.expr("try_cast(RecipeId as int)"))
    .withColumn("Rating", F.expr("try_cast(Rating as double)"))
    .dropna(subset=["RecipeId", "Rating"])
)

## 7. ดูสถิติเบื้องต้นด้วย describe

In [72]:
recipes_clean.describe(["Calories"]).show()

+-------+------------------+
|summary|          Calories|
+-------+------------------+
|  count|            522517|
|   mean|484.43857998878127|
| stddev|1397.1166490876112|
|    min|               0.0|
|    max|          612854.6|
+-------+------------------+

+-------+------------------+
|summary|          Calories|
+-------+------------------+
|  count|            522517|
|   mean|484.43857998878127|
| stddev|1397.1166490876112|
|    min|               0.0|
|    max|          612854.6|
+-------+------------------+



- `describe(["Calories"])`  
  ดูสถิติของ column `Calories`


In [73]:
reviews_clean.describe(["Rating"]).show()

+-------+------------------+
|summary|            Rating|
+-------+------------------+
|  count|           1401982|
|   mean| 4.407951029328479|
| stddev|1.2720116809642992|
|    min|               0.0|
|    max|               5.0|
+-------+------------------+

+-------+------------------+
|summary|            Rating|
+-------+------------------+
|  count|           1401982|
|   mean| 4.407951029328479|
| stddev|1.2720116809642992|
|    min|               0.0|
|    max|               5.0|
+-------+------------------+



ใช้ดูภาพรวมคะแนน review เช่น ค่าเฉลี่ย ต่ำสุด สูงสุด

## 8. นับค่าที่ไม่ซ้ำด้วย distinct

In [74]:
recipes_clean.select("RecipeCategory").distinct().count()

312

312

- `select("RecipeCategory")`  
  เลือกเฉพาะ column หมวดหมู่

- `distinct()`  
  เอาค่าที่ไม่ซ้ำ

- `count()`  
  นับจำนวน category ที่ไม่ซ้ำ

In [75]:
reviews_clean.select("RecipeId").distinct().count()

271678

271678

ใช้ดูว่าใน review มีสูตรอาหารกี่สูตรที่ถูกรีวิวอย่างน้อย 1 ครั้ง

## 9. Filter


In [76]:
high_rating_reviews = reviews_clean.filter(F.col("Rating") >= 4)
high_rating_reviews.show(5)

+--------+--------+------+--------------------+--------------------+
|RecipeId|AuthorId|Rating|              Review|       DateSubmitted|
+--------+--------+------+--------------------+--------------------+
|     992|    2008|   5.0|better than any y...|2000-01-25T21:44:00Z|
|    4384|    1634|   4.0|I cut back on the...|2001-10-17T16:49:59Z|
|    7435|    1773|   5.0|easily the best i...|2000-03-13T21:15:00Z|
|      44|    2085|   5.0|  An excellent dish.|2000-03-28T12:51:00Z|
|    5221|    2046|   4.0|love it, but with...|2000-05-08T11:08:00Z|
+--------+--------+------+--------------------+--------------------+
only showing top 5 rows
+--------+--------+------+--------------------+--------------------+
|RecipeId|AuthorId|Rating|              Review|       DateSubmitted|
+--------+--------+------+--------------------+--------------------+
|     992|    2008|   5.0|better than any y...|2000-01-25T21:44:00Z|
|    4384|    1634|   4.0|I cut back on the...|2001-10-17T16:49:59Z|
|    7435|

- `filter(F.col("Rating") >= 4)`  
  เลือกเฉพาะ review ที่ rating มากกว่าหรือเท่ากับ 4

- `show(5)`  
  แสดงผล 5 แถวแรก

## 10. when / otherwise

> สร้าง column จากเงื่อนไข



In [77]:
reviews_labeled = reviews_clean.withColumn(
    "rating_label",
    F.when(F.col("Rating") >= 4, "good")
     .when(F.col("Rating") >= 2, "normal")
     .otherwise("bad")
)

reviews_labeled.select("RecipeId", "Rating", "rating_label").show(10)

+--------+------+------------+
|RecipeId|Rating|rating_label|
+--------+------+------------+
|     992|   5.0|        good|
|    4384|   4.0|        good|
|    4523|   2.0|      normal|
|    7435|   5.0|        good|
|      44|   5.0|        good|
|    5221|   4.0|        good|
|   13307|   5.0|        good|
|     148|   0.0|         bad|
|     517|   5.0|        good|
|    4684|   5.0|        good|
+--------+------+------------+
only showing top 10 rows
+--------+------+------------+
|RecipeId|Rating|rating_label|
+--------+------+------------+
|     992|   5.0|        good|
|    4384|   4.0|        good|
|    4523|   2.0|      normal|
|    7435|   5.0|        good|
|      44|   5.0|        good|
|    5221|   4.0|        good|
|   13307|   5.0|        good|
|     148|   0.0|         bad|
|     517|   5.0|        good|
|    4684|   5.0|        good|
+--------+------+------------+
only showing top 10 rows


- `withColumn("rating_label", ...)`  
  สร้าง column ใหม่ชื่อ `rating_label`

- `F.when(F.col("Rating") >= 4, "good")`  
  ถ้า rating >= 4 ให้เป็น good

- `.when(F.col("Rating") >= 2, "normal")`  
  ถ้า rating >= 2 ให้เป็น normal

- `.otherwise("bad")`  
  ถ้าไม่เข้าเงื่อนไขก่อนหน้า ให้เป็น bad



## 11. like / contains / startswith / endswith ใช้ค้นหาข้อความ

In [78]:
recipes_clean.select(
    "Name",
    F.col("Name").like("%Chicken%").alias("has_chicken_in_name")
).show(10, truncate=False)

+------------------------------+-------------------+
|Name                          |has_chicken_in_name|
+------------------------------+-------------------+
|Best Lemonade                 |false              |
|Carina's Tofu-Vegetable Kebabs|false              |
|Best Blackbottom Pie          |false              |
|Warm Chicken A La King        |true               |
|Butter Pecan Cookies          |false              |
|Boston Cream Pie              |false              |
|Cafe Cappuccino               |false              |
|Jimmy G's Carrot Cake         |false              |
|Carrot Cake                   |false              |
|Black Bean Salsa              |false              |
+------------------------------+-------------------+
only showing top 10 rows
+------------------------------+-------------------+
|Name                          |has_chicken_in_name|
+------------------------------+-------------------+
|Best Lemonade                 |false              |
|Carina's Tofu-Vegeta

- `like("%Chicken%")`  
  ตรวจว่าชื่อเมนูมีคำว่า Chicken หรือไม่

- `%`  
  หมายถึงมีตัวอักษรอะไรก่อนหรือหลังก็ได้

In [79]:
recipes_clean.filter(
    F.lower(F.col("RecipeIngredientParts")).contains("chicken")
).select(
    "Name",
    "RecipeIngredientParts"
).show(10, truncate=False)

+-------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Name                                             |RecipeIngredientParts                                                                                                                                                                                                                                                                                                                                  |
+-------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------

- `F.lower(...)`  
  แปลงข้อความเป็นตัวเล็กก่อนค้นหา

- `.contains("chicken")`  
  เช็กว่ามีคำว่า "chicken" อยู่ใน ingredient หรือไม่

In [80]:
recipes_clean.select(
    "Name",
    F.col("Name").startswith("Chicken").alias("start_with_chicken"),
    F.col("Name").endswith("Soup").alias("end_with_soup")
).show(10, truncate=False)

+------------------------------+------------------+-------------+
|Name                          |start_with_chicken|end_with_soup|
+------------------------------+------------------+-------------+
|Best Lemonade                 |false             |false        |
|Carina's Tofu-Vegetable Kebabs|false             |false        |
|Best Blackbottom Pie          |false             |false        |
|Warm Chicken A La King        |false             |false        |
|Butter Pecan Cookies          |false             |false        |
|Boston Cream Pie              |false             |false        |
|Cafe Cappuccino               |false             |false        |
|Jimmy G's Carrot Cake         |false             |false        |
|Carrot Cake                   |false             |false        |
|Black Bean Salsa              |false             |false        |
+------------------------------+------------------+-------------+
only showing top 10 rows
+------------------------------+------------------+

- `startswith("Chicken")`  
  เช็กว่าชื่อเมนูขึ้นต้นด้วย Chicken หรือไม่

- `endswith("Soup")`  
  เช็กว่าชื่อเมนูลงท้ายด้วย Soup หรือไม่

## 12. substr
> ตัดข้อความบางส่วน



In [81]:
recipes_clean.select(
    "Name",
    F.col("Name").substr(1, 10).alias("short_name")
).show(10, truncate=False)

+------------------------------+----------+
|Name                          |short_name|
+------------------------------+----------+
|Best Lemonade                 |Best Lemon|
|Carina's Tofu-Vegetable Kebabs|Carina's T|
|Best Blackbottom Pie          |Best Black|
|Warm Chicken A La King        |Warm Chick|
|Butter Pecan Cookies          |Butter Pec|
|Boston Cream Pie              |Boston Cre|
|Cafe Cappuccino               |Cafe Cappu|
|Jimmy G's Carrot Cake         |Jimmy G's |
|Carrot Cake                   |Carrot Cak|
|Black Bean Salsa              |Black Bean|
+------------------------------+----------+
only showing top 10 rows
+------------------------------+----------+
|Name                          |short_name|
+------------------------------+----------+
|Best Lemonade                 |Best Lemon|
|Carina's Tofu-Vegetable Kebabs|Carina's T|
|Best Blackbottom Pie          |Best Black|
|Warm Chicken A La King        |Warm Chick|
|Butter Pecan Cookies          |Butter Pec|
|Boston

- `substr(1, 10)`  
  ตัดข้อความจากตำแหน่งที่ 1 จำนวน 10 ตัวอักษร

- `alias("short_name")`  
  ตั้งชื่อ column ใหม่

## 13. between เลือกค่าที่อยู่ในช่วง

In [82]:
recipes_clean.filter(
    F.col("Calories").isNotNull()
).select(
    "Name",
    "Calories",
    F.col("Calories").between(100, 500).alias("calories_100_to_500")
).show(10, truncate=False)

+------------------------------+--------+-------------------+
|Name                          |Calories|calories_100_to_500|
+------------------------------+--------+-------------------+
|Best Lemonade                 |311.1   |true               |
|Carina's Tofu-Vegetable Kebabs|536.1   |false              |
|Best Blackbottom Pie          |437.9   |true               |
|Warm Chicken A La King        |895.5   |false              |
|Butter Pecan Cookies          |69.0    |false              |
|Boston Cream Pie              |688.2   |false              |
|Cafe Cappuccino               |62.2    |false              |
|Jimmy G's Carrot Cake         |372.9   |true               |
|Carrot Cake                   |522.6   |false              |
|Black Bean Salsa              |114.3   |true               |
+------------------------------+--------+-------------------+
only showing top 10 rows
+------------------------------+--------+-------------------+
|Name                          |Calories|calo

- `between(100, 500)`  
  เช็กว่า calories อยู่ระหว่าง 100 ถึง 500 หรือไม่

ผลลัพธ์เป็น boolean


In [83]:
recipes_clean.filter(
    F.col("Calories").between(100, 500)
).select(
    "Name",
    "Calories"
).show(10, truncate=False)

+------------------------------+--------+
|Name                          |Calories|
+------------------------------+--------+
|Best Lemonade                 |311.1   |
|Best Blackbottom Pie          |437.9   |
|Jimmy G's Carrot Cake         |372.9   |
|Black Bean Salsa              |114.3   |
|Almond Pound Cake             |408.5   |
|Brownie Pudding               |182.8   |
|Alfredo Sauce                 |489.9   |
|Buttermilk Ranch Dressing     |267.2   |
|Butter Madeira Cake           |373.8   |
|Brown Rice and Vegetable Pilaf|412.6   |
+------------------------------+--------+
only showing top 10 rows
+------------------------------+--------+
|Name                          |Calories|
+------------------------------+--------+
|Best Lemonade                 |311.1   |
|Best Blackbottom Pie          |437.9   |
|Jimmy G's Carrot Cake         |372.9   |
|Black Bean Salsa              |114.3   |
|Almond Pound Cake             |408.5   |
|Brownie Pudding               |182.8   |
|Alfredo 

ใช้ `between` เป็นเงื่อนไขใน `filter`  
เพื่อเอาเฉพาะสูตรที่ calories อยู่ในช่วง 100 ถึง 500

## 14. เพิ่ม ลบ เปลี่ยนชื่อ Column

In [84]:
recipes_with_calorie_group = recipes_clean.withColumn(
    "calorie_group",
    F.when(F.col("Calories") < 200, "low")
     .when(F.col("Calories") < 500, "medium")
     .otherwise("high")
)

recipes_with_calorie_group.select(
    "Name",
    "Calories",
    "calorie_group"
).show(10, truncate=False)

+------------------------------+--------+-------------+
|Name                          |Calories|calorie_group|
+------------------------------+--------+-------------+
|Best Lemonade                 |311.1   |medium       |
|Carina's Tofu-Vegetable Kebabs|536.1   |high         |
|Best Blackbottom Pie          |437.9   |medium       |
|Warm Chicken A La King        |895.5   |high         |
|Butter Pecan Cookies          |69.0    |low          |
|Boston Cream Pie              |688.2   |high         |
|Cafe Cappuccino               |62.2    |low          |
|Jimmy G's Carrot Cake         |372.9   |medium       |
|Carrot Cake                   |522.6   |high         |
|Black Bean Salsa              |114.3   |low          |
+------------------------------+--------+-------------+
only showing top 10 rows
+------------------------------+--------+-------------+
|Name                          |Calories|calorie_group|
+------------------------------+--------+-------------+
|Best Lemonade         

- `withColumn("calorie_group", ...)`  
  เพิ่ม column ใหม่

- ถ้า Calories < 200 ให้เป็น low
- ถ้า Calories < 500 ให้เป็น medium
- นอกนั้นให้เป็น high

In [85]:
recipes_drop_example = recipes_with_calorie_group.drop("CookTime", "PrepTime")
recipes_drop_example.columns

['RecipeId',
 'Name',
 'RecipeCategory',
 'TotalTime',
 'RecipeServings',
 'RecipeIngredientParts',
 'Calories',
 'calorie_group']

['RecipeId',
 'Name',
 'RecipeCategory',
 'TotalTime',
 'RecipeServings',
 'RecipeIngredientParts',
 'Calories',
 'calorie_group']

- `drop("CookTime", "PrepTime")`  
  ลบ column ที่ไม่ต้องการออก

ไม่ได้ลบจาก DataFrame เดิม แต่สร้าง DataFrame ใหม่

In [86]:
recipes_rename_example = recipes_with_calorie_group.withColumnRenamed(
    "RecipeCategory",
    "Category"
)

recipes_rename_example.columns

['RecipeId',
 'Name',
 'Category',
 'CookTime',
 'PrepTime',
 'TotalTime',
 'RecipeServings',
 'RecipeIngredientParts',
 'Calories',
 'calorie_group']

['RecipeId',
 'Name',
 'Category',
 'CookTime',
 'PrepTime',
 'TotalTime',
 'RecipeServings',
 'RecipeIngredientParts',
 'Calories',
 'calorie_group']

- `withColumnRenamed("RecipeCategory", "Category")`  
  เปลี่ยนชื่อ column จาก `RecipeCategory` เป็น `Category`

## 15. GroupBy + Aggregation


In [87]:
rating_by_recipe = (
    reviews_clean
    .groupBy("RecipeId")
    .agg(
        F.count("*").alias("review_count"),
        F.avg("Rating").alias("avg_rating")
    )
)

- `reviews_clean.groupBy("RecipeId")`  
  รวมข้อมูลรีวิวทั้งหมดโดยจัดกลุ่มตาม `RecipeId` เพื่อให้รีวิวของสูตรเดียวกันอยู่ในกลุ่มเดียวกัน

- `F.count("*")`  
  นับจำนวนรีวิวทั้งหมดในแต่ละกลุ่ม หรือแต่ละสูตรอาหาร

- `.alias("review_count")`  
  ตั้งชื่อ column ใหม่ให้ผลลัพธ์ของการนับว่า `review_count`

- `F.avg("Rating")`  
  คำนวณค่าเฉลี่ยของคะแนน `Rating` ในแต่ละกลุ่ม เพื่อหาคะแนนเฉลี่ยของสูตรนั้น

- `.alias("avg_rating")`  
  ตั้งชื่อ column ใหม่ให้ผลลัพธ์ของค่าเฉลี่ยว่า `avg_rating`

ผลลัพธ์:
- 1 row ต่อ 1 `RecipeId`
- แต่ละสูตรจะมี
  - จำนวนรีวิว (`review_count`)
  - คะแนนเฉลี่ย (`avg_rating`)

In [88]:
rating_by_recipe.orderBy(F.desc("review_count")).show(10)

+--------+------------+------------------+
|RecipeId|review_count|        avg_rating|
+--------+------------+------------------+
|   45809|        2892| 4.314661134163209|
|    2886|        2182| 4.218148487626031|
|   27208|        1614| 4.281908302354399|
|   89204|        1584|4.2228535353535355|
|   39087|        1491| 4.538564721663313|
|   67256|        1359| 4.325974981604121|
|   35813|        1353|4.4368070953436805|
|   54257|        1325| 4.212830188679245|
|   22782|        1273| 4.423409269442263|
|   32204|        1228| 4.521986970684039|
+--------+------------+------------------+
only showing top 10 rows
+--------+------------+------------------+
|RecipeId|review_count|        avg_rating|
+--------+------------+------------------+
|   45809|        2892| 4.314661134163209|
|    2886|        2182| 4.218148487626031|
|   27208|        1614| 4.281908302354399|
|   89204|        1584|4.2228535353535355|
|   39087|        1491| 4.538564721663313|
|   67256|        1359| 4.325

- `orderBy(F.desc("review_count"))`  
  เรียงจากสูตรที่มีจำนวน review มากที่สุดไปน้อยที่สุด

- `show(10)`  
  แสดง 10 แถวแรก

## 16. Join

In [89]:
recipe_rating = (
    recipes_clean
    .join(rating_by_recipe, on="RecipeId", how="left")
)

- `recipes_clean.join(...)`  
  รวมข้อมูลสูตรอาหารกับข้อมูลคะแนนรีวิว

- `rating_by_recipe`  
  DataFrame ที่มีจำนวนรีวิวและคะแนนเฉลี่ยของแต่ละสูตร

- `on="RecipeId"`  
  ใช้ `RecipeId` เป็น key ในการ join

- `how="left"`  
  เก็บสูตรอาหารทั้งหมดจาก `recipes_clean` ไว้ ถึงแม้บางสูตรจะไม่มี review

ผลลัพธ์:
- ได้ DataFrame ใหม่ชื่อ `recipe_rating`
- แต่ละสูตรจะมีข้อมูล recipe + จำนวน review + คะแนนเฉลี่ย

In [90]:
recipe_rating.select(
    "RecipeId",
    "Name",
    "RecipeCategory",
    "Calories",
    "review_count",
    "avg_rating"
).show(10, truncate=False)

+--------+------------------------------+--------------+--------+------------+-----------------+
|RecipeId|Name                          |RecipeCategory|Calories|review_count|avg_rating       |
+--------+------------------------------+--------------+--------+------------+-----------------+
|40      |Best Lemonade                 |Beverages     |311.1   |9           |4.333333333333333|
|41      |Carina's Tofu-Vegetable Kebabs|Soy/Tofu      |536.1   |2           |4.5              |
|43      |Best Blackbottom Pie          |Pie           |437.9   |1           |1.0              |
|44      |Warm Chicken A La King        |Chicken       |895.5   |22          |4.545454545454546|
|47      |Butter Pecan Cookies          |Dessert       |69.0    |2           |4.0              |
|48      |Boston Cream Pie              |Pie           |688.2   |2           |1.0              |
|52      |Cafe Cappuccino               |Beverages     |62.2    |1           |5.0              |
|53      |Jimmy G's Carrot Cak

- `recipe_rating.select(...)`  
  เลือกเฉพาะ column ที่ต้องการแสดงจาก DataFrame `recipe_rating`

- `.show(10, truncate=False)`  
  แสดงข้อมูล 10 แถวแรก โดย `truncate=False` ทำให้ข้อความยาว ๆ ไม่ถูกตัด

## 17. วิเคราะห์สูตรที่ popular และ rating สูง

In [91]:
popular_high_rated = (
    recipe_rating
    .filter(F.col("review_count") >= 50)
    .filter(F.col("avg_rating") >= 4.5)
    .orderBy(F.desc("avg_rating"), F.desc("review_count"))
)

popular_high_rated.select(
    "Name",
    "RecipeCategory",
    "avg_rating",
    "review_count",
    "Calories"
).show(20, truncate=False)

+-------------------------------------------------------+--------------+------------------+------------+--------+
|Name                                                   |RecipeCategory|avg_rating        |review_count|Calories|
+-------------------------------------------------------+--------------+------------------+------------+--------+
|Caprese Salad Tomatoes (Italian Marinated Tomatoes)    |Vegetable     |5.0               |52          |137.5   |
|Mexican Stack-Up #RSC                                  |Black Beans   |4.990783410138249 |217         |793.0   |
|Syrup for Blueberry Pancakes                           |Sauces        |4.964912280701754 |57          |376.6   |
|Toffee Dip with Apples                                 |Lunch/Snacks  |4.963636363636364 |55          |1192.6  |
|Make Your Own Boursin Cheese - Paula Deen              |Spreads       |4.962962962962963 |54          |1065.2  |
|Mean's Lamb You Can Eat With a Spoon                   |Lamb/Sheep    |4.96            

- `filter(F.col("review_count") >= 50)`  
  เลือกสูตรที่มีรีวิวอย่างน้อย 50 ครั้ง

- `filter(F.col("avg_rating") >= 4.5)`  
  เลือกสูตรที่คะแนนเฉลี่ยอย่างน้อย 4.5

- `orderBy(F.desc("avg_rating"), F.desc("review_count"))`  
  เรียงจากคะแนนสูงก่อน แล้วค่อยดูจำนวน review

## 18. วิเคราะห์ตาม Category

In [92]:
category_summary = (
    recipe_rating
    .groupBy("RecipeCategory")
    .agg(
        F.count("*").alias("recipe_count"),
        F.avg("avg_rating").alias("category_avg_rating"),
        F.avg("Calories").alias("avg_calories")
    )
    .orderBy(F.desc("recipe_count"))
)

category_summary.show(20, truncate=False)

+--------------+------------+-------------------+------------------+
|RecipeCategory|recipe_count|category_avg_rating|avg_calories      |
+--------------+------------+-------------------+------------------+
|Dessert       |62072       |4.24364920661682   |643.8104233792989 |
|Lunch/Snacks  |32586       |4.403159714035769  |437.65264530780127|
|One Dish Meal |31345       |4.281247546294441  |529.7373329079594 |
|Vegetable     |27231       |4.41054181697956   |271.88961110499264|
|Breakfast     |21101       |4.353555405940915  |382.85989289607204|
|Beverages     |16076       |4.521560677502436  |283.1140333416283 |
|Chicken       |13249       |4.360758863622481  |536.77809646011   |
|Meat          |13131       |4.283458514874751  |606.2567511994523 |
|Breads        |12804       |4.248610480403898  |618.7491487035314 |
|Pork          |12603       |4.375726736875861  |558.1378878044925 |
|Sauces        |12166       |4.384590219865782  |384.4776837086968 |
|Chicken Breast|11282       |4.376

- `groupBy("RecipeCategory")`  
  จัดกลุ่มข้อมูลตาม"หมวดหมู่อาหาร"

- `F.count("*").alias("recipe_count")`  
  นับจำนวนสูตรในแต่ละหมวด

- `F.avg("avg_rating").alias("category_avg_rating")`  
  คำนวณ rating เฉลี่ยของหมวดนั้น

- `F.avg("Calories").alias("avg_calories")`  
  คำนวณ calories เฉลี่ยของหมวดนั้น

## 19. Sort / OrderBy

In [93]:
recipe_rating.sort(F.col("avg_rating").desc()).show(10, truncate=False)

+--------+----------------------------------------------+--------------+--------+--------+---------+--------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+------------+----------+
|RecipeId|Name                                          |RecipeCategory|CookTime|PrepTime|TotalTime|RecipeServings|RecipeIngredientParts                                                                                                                                                                                     |Calories|review_count|avg_rating|
+--------+----------------------------------------------+--------------+--------+--------+---------+--------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

- `sort(F.col("avg_rating").desc())`  
  เรียงจากคะแนนเฉลี่ยมากไปน้อย

In [94]:
recipe_rating.orderBy(
    F.col("review_count").desc(),
    F.col("avg_rating").desc()
).show(10, truncate=False)

+--------+-----------------------------------------------------+--------------+--------+--------+---------+--------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+------------+------------------+
|RecipeId|Name                                                 |RecipeCategory|CookTime|PrepTime|TotalTime|RecipeServings|RecipeIngredientParts                                                                                                                                                                                                                                            |Calories|review_count|avg_rating        |
+--------+-----------------------------------------------------+--------------+--------+--------+---------+--------------+----------------------------------

- `orderBy(...)`  
  เรียงได้หลาย column

อันนี้จะเรียงจาก
1. เรียงจาก review เยอะสุดก่อน
2. ถ้า review เท่ากัน เรียงจาก rating สูงสุด

## 20. Cache เมื่อใช้ DataFrame ซ้ำ

In [95]:
recipe_rating.cache()
recipe_rating.count()

522517

522517

- `cache()`  
  บอก Spark ว่า DataFrame นี้อาจถูกใช้ซ้ำ ให้เก็บไว้ใน memory

- `count()`  
  เป็น Action ที่ทำให้ Spark compute จริง และเริ่ม cache ข้อมูล

เหมาะกับ DataFrame ที่ใช้ซ้ำหลายรอบ

## 21. Convert DataFrame เป็น RDD / JSON / Pandas

In [96]:
recipe_rdd = recipe_rating.rdd

- `.rdd`  
  แปลง Spark DataFrame เป็น RDD

ปัจจุบันใช้ DataFrame มากกว่า เพราะใช้งานง่ายและ Spark optimize การทำงานให้อัตโนมัติ แต่ RDD คือโครงสร้างพื้นฐานระดับล่างของ Spark ที่ควรรู้ไว้


In [97]:
recipe_rating.toJSON().first()

'{"RecipeId":148,"Name":"Bugwiches","RecipeCategory":"Lunch/Snacks","CookTime":"NA","PrepTime":"PT40M","TotalTime":"PT40M","RecipeServings":"1","RecipeIngredientParts":"c(\\"carrot\\", \\"lettuce\\", \\"cucumber\\", \\"radish\\", \\"raisins\\", \\"mayonnaise\\", \\"celery\\", \\"tuna salad\\", \\"salmon salad\\")","Calories":84.0,"review_count":1,"avg_rating":0.0}'

'{"RecipeId":148,"Name":"Bugwiches","RecipeCategory":"Lunch/Snacks","CookTime":"NA","PrepTime":"PT40M","TotalTime":"PT40M","RecipeServings":"1","RecipeIngredientParts":"c(\\"carrot\\", \\"lettuce\\", \\"cucumber\\", \\"radish\\", \\"raisins\\", \\"mayonnaise\\", \\"celery\\", \\"tuna salad\\", \\"salmon salad\\")","Calories":84.0,"review_count":1,"avg_rating":0.0}'



- `toJSON()`  
  แปลงแต่ละ row เป็น JSON string

- `first()`  
  ดู JSON แถวแรก

In [98]:
small_pandas_df = recipe_rating.limit(10).toPandas()
small_pandas_df

,RecipeId,Name,RecipeCategory,CookTime,PrepTime,TotalTime,RecipeServings,RecipeIngredientParts,Calories,review_count,avg_rating
0,148,Bugwiches,Lunch/Snacks,NA,PT40M,PT40M,1,"c(""carrot"", ""lettuce"", ""cucumber"", ""radish"", ""...",84.0,1.0,0.000000
1,463,Cappuccino Coffee Mix,Beverages,NA,PT20M,PT20M,15,"c(""instant coffee granules"", ""granulated sugar...",54.3,7.0,3.714286
2,471,Seared Salmon with Horseradish Tomato Vinaigrette,Summer,PT9M,PT1H10M,PT1H19M,6,"c(""salmon fillets"", ""canola oil"", ""butter"", ""r...",797.3,3.0,3.666667
3,496,7-Up Cake,Dessert,PT42M,PT45M,PT1H27M,12,"c(""pineapple instant pudding"", ""eggs"", ""margar...",550.2,1.0,5.000000
4,833,Hamantaschen-Cookies (Haman's Hats),Dessert,PT8M,PT0S,PT8M,NA,"c(""eggs"", ""sugar"", ""orange zest"", ""margarine"",...",89.8,2.0,2.500000
5,1580,Dutch Buttercake,Dessert,PT10M,PT35M,PT45M,10,"c(""flour"", ""baking powder"", ""sugar"", ""unsalted...",312.7,2.0,4.500000
6,1591,Grilled Lemon Chicken,Chicken Breast,PT15M,PT45M,PT1H,4,"c(""fresh lemon juice"", ""extra virgin olive oil...",123.2,4.0,4.250000
7,1959,Sylvia's Famous Spareribs,< 15 Mins,NA,PT0S,PT0S,8,"c(""salt"", ""black pepper"", ""red pepper flakes"",...",982.3,NaN,NaN
8,2122,English Cream Scones,Scones,NA,PT0S,PT0S,NA,"c(""flour"", ""sugar"", ""baking powder"", ""salt"", ""...",211.5,2.0,4.000000
9,2142,Buttery Drop Cookies,Drop Cookies,PT7M,PT0S,PT7M,NA,"c(""sugar"", ""milk"", ""egg"", ""vanilla"", ""all-purp...",60.0,1.0,5.000000


,RecipeId,Name,RecipeCategory,CookTime,PrepTime,TotalTime,RecipeServings,RecipeIngredientParts,Calories,review_count,avg_rating
0,148,Bugwiches,Lunch/Snacks,NA,PT40M,PT40M,1,"c(""carrot"", ""lettuce"", ""cucumber"", ""radish"", ""...",84.0,1.0,0.000000
1,463,Cappuccino Coffee Mix,Beverages,NA,PT20M,PT20M,15,"c(""instant coffee granules"", ""granulated sugar...",54.3,7.0,3.714286
2,471,Seared Salmon with Horseradish Tomato Vinaigrette,Summer,PT9M,PT1H10M,PT1H19M,6,"c(""salmon fillets"", ""canola oil"", ""butter"", ""r...",797.3,3.0,3.666667
3,496,7-Up Cake,Dessert,PT42M,PT45M,PT1H27M,12,"c(""pineapple instant pudding"", ""eggs"", ""margar...",550.2,1.0,5.000000
4,833,Hamantaschen-Cookies (Haman's Hats),Dessert,PT8M,PT0S,PT8M,NA,"c(""eggs"", ""sugar"", ""orange zest"", ""margarine"",...",89.8,2.0,2.500000
5,1580,Dutch Buttercake,Dessert,PT10M,PT35M,PT45M,10,"c(""flour"", ""baking powder"", ""sugar"", ""unsalted...",312.7,2.0,4.500000
6,1591,Grilled Lemon Chicken,Chicken Breast,PT15M,PT45M,PT1H,4,"c(""fresh lemon juice"", ""extra virgin olive oil...",123.2,4.0,4.250000
7,1959,Sylvia's Famous Spareribs,< 15 Mins,NA,PT0S,PT0S,8,"c(""salt"", ""black pepper"", ""red pepper flakes"",...",982.3,NaN,NaN
8,2122,English Cream Scones,Scones,NA,PT0S,PT0S,NA,"c(""flour"", ""sugar"", ""baking powder"", ""salt"", ""...",211.5,2.0,4.000000
9,2142,Buttery Drop Cookies,Drop Cookies,PT7M,PT0S,PT7M,NA,"c(""sugar"", ""milk"", ""egg"", ""vanilla"", ""all-purp...",60.0,1.0,5.000000


- `limit(10)`  
  จำกัดข้อมูลเหลือ 10 แถวก่อน

- `toPandas()`  
  ดึงข้อมูลจาก Spark มาเป็น Pandas DataFrame



## 22. Save Output เป็น Parquet / CSV / JSON

In [99]:
popular_high_rated.write.mode("overwrite").parquet(
    "output/popular_high_rated_recipes.parquet"
)

- `.write`  
  เริ่มเขียน DataFrame ออกเป็นไฟล์

- `.mode("overwrite")`  
  ถ้ามี output เดิม ให้เขียนทับ

- `.parquet(...)`  
  บันทึกเป็น Parquet format

Parquet เหมาะกับ Spark เพราะอ่านเร็วและเก็บ schema ได้

In [100]:
category_summary.write.mode("overwrite").csv(
    "output/category_summary.csv",
    header=True
)

- `.csv(...)`  
  บันทึกเป็น CSV

- `header=True`  
  ให้ไฟล์มีชื่อ column

CSV เหมาะกับการเปิดดูใน Excel หรือ Google Sheets

In [101]:
category_summary.write.mode("overwrite").json(
    "output/category_summary_json"
)

- `.json(...)`  
  บันทึก output เป็น JSON

เหมาะกับงานที่ต้องส่งต่อข้อมูลให้ระบบอื่น

## 23. Spark SQL

In [102]:
recipe_rating.createOrReplaceTempView("recipe_rating")

- `createOrReplaceTempView("recipe_rating")`  
  สร้าง temporary view จาก DataFrame

เพื่อที่หลังจากนี้สามารถใช้ SQL query กับ view นี้ได้

In [103]:
spark.sql("""
SELECT
    RecipeCategory,
    COUNT(*) AS recipe_count,
    AVG(avg_rating) AS category_avg_rating
FROM recipe_rating
GROUP BY RecipeCategory
ORDER BY recipe_count DESC
""").show(20, truncate=False)

+--------------+------------+-------------------+
|RecipeCategory|recipe_count|category_avg_rating|
+--------------+------------+-------------------+
|Dessert       |62072       |4.243649206616812  |
|Lunch/Snacks  |32586       |4.403159714035772  |
|One Dish Meal |31345       |4.281247546294448  |
|Vegetable     |27231       |4.410541816979569  |
|Breakfast     |21101       |4.353555405940917  |
|Beverages     |16076       |4.521560677502439  |
|Chicken       |13249       |4.3607588636224826 |
|Meat          |13131       |4.283458514874751  |
|Breads        |12804       |4.2486104804039    |
|Pork          |12603       |4.375726736875868  |
|Sauces        |12166       |4.384590219865782  |
|Chicken Breast|11282       |4.376158489752838  |
|Potato        |10870       |4.424074260361769  |
|Quick Breads  |10387       |4.226220343603106  |
|< 60 Mins     |9719        |4.331251186852882  |
|< 30 Mins     |9020        |4.404099383190222  |
|Cheese        |8462        |4.394971679293944  |


- `spark.sql(...)`  
  ใช้เขียน SQL บน Spark

- `GROUP BY RecipeCategory`  
  จัดกลุ่มตาม "หมวดหมู่อาหาร"

- `COUNT(*)`  
  นับจำนวนสูตร

- `AVG(avg_rating)`  
  คำนวณคะแนนเฉลี่ยของหมวดนั้น

## 24.Execution Plan

In [104]:
popular_high_rated.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [avg_rating#1122 DESC NULLS LAST, review_count#1121L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(avg_rating#1122 DESC NULLS LAST, review_count#1121L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=3100]
      +- Filter (((isnotnull(review_count#1121L) AND isnotnull(avg_rating#1122)) AND (review_count#1121L >= 50)) AND (avg_rating#1122 >= 4.5))
         +- InMemoryTableScan [RecipeId#542, Name#1, RecipeCategory#10, CookTime#4, PrepTime#5, TotalTime#6, RecipeServings#25, RecipeIngredientParts#13, Calories#544, review_count#1121L, avg_rating#1122], [isnotnull(review_count#1121L), isnotnull(avg_rating#1122), (review_count#1121L >= 50), (avg_rating#1122 >= 4.5)]
               +- InMemoryRelation [RecipeId#542, Name#1, RecipeCategory#10, CookTime#4, PrepTime#5, TotalTime#6, RecipeServings#25, RecipeIngredientParts#13, Calories#544, review_count#1121L, avg_rating#1122], StorageLevel(disk, memory, deserialized

- `explain()`  
  ใช้ดูว่า Spark วางแผน execute งานอย่างไร

เป้าหมายเพื่อดูว่า Spark ไม่ได้ run ทีละบรรทัดทันที แต่จะสร้าง plan แล้วค่อย run ตอนเจอ Action

## 25. Scala Notes เบื้องต้น



In [105]:
# Scala note: Start SparkSession

'''
import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder()
  .appName("FoodCom Spark Practice")
  .getOrCreate()
'''

'\nimport org.apache.spark.sql.SparkSession\n\nval spark = SparkSession.builder()\n  .appName("FoodCom Spark Practice")\n  .getOrCreate()\n'

'\nimport org.apache.spark.sql.SparkSession\n\nval spark = SparkSession.builder()\n  .appName("FoodCom Spark Practice")\n  .getOrCreate()\n'

In [106]:
# Scala note: Read CSV

'''
val recipesDF = spark.read
  .option("header", "true")
  .csv("recipes.csv")
'''

'\nval recipesDF = spark.read\n  .option("header", "true")\n  .csv("recipes.csv")\n'

'\nval recipesDF = spark.read\n  .option("header", "true")\n  .csv("recipes.csv")\n'

In [107]:
# Scala note: Select + Filter

'''
import spark.implicits._

val recipesSmall = recipesDF
  .select("RecipeId", "Name", "RecipeCategory", "Calories")
  .filter($"Calories" > 0)
'''

'\nimport spark.implicits._\n\nval recipesSmall = recipesDF\n  .select("RecipeId", "Name", "RecipeCategory", "Calories")\n  .filter($"Calories" > 0)\n'

'\nimport spark.implicits._\n\nval recipesSmall = recipesDF\n  .select("RecipeId", "Name", "RecipeCategory", "Calories")\n  .filter($"Calories" > 0)\n'

In [108]:
# Scala note: GroupBy + Aggregation

'''
import org.apache.spark.sql.functions._

val ratingByRecipe = reviewsDF
  .groupBy("RecipeId")
  .agg(
    count("*").alias("review_count"),
    avg("Rating").alias("avg_rating")
  )
'''

'\nimport org.apache.spark.sql.functions._\n\nval ratingByRecipe = reviewsDF\n  .groupBy("RecipeId")\n  .agg(\n    count("*").alias("review_count"),\n    avg("Rating").alias("avg_rating")\n  )\n'

'\nimport org.apache.spark.sql.functions._\n\nval ratingByRecipe = reviewsDF\n  .groupBy("RecipeId")\n  .agg(\n    count("*").alias("review_count"),\n    avg("Rating").alias("avg_rating")\n  )\n'

In [109]:
# Scala note: Join

'''
val recipeRating = recipesDF
  .join(ratingByRecipe, Seq("RecipeId"), "left")
'''

'\nval recipeRating = recipesDF\n  .join(ratingByRecipe, Seq("RecipeId"), "left")\n'

'\nval recipeRating = recipesDF\n  .join(ratingByRecipe, Seq("RecipeId"), "left")\n'

In [110]:
# Scala note: Save Output

'''
recipeRating.write
  .mode("overwrite")
  .parquet("output/recipe_rating.parquet")
'''

'\nrecipeRating.write\n  .mode("overwrite")\n  .parquet("output/recipe_rating.parquet")\n'

'\nrecipeRating.write\n  .mode("overwrite")\n  .parquet("output/recipe_rating.parquet")\n'

สรุป Scala

- `val`  
  ใช้ประกาศตัวแปรที่ reference ไม่เปลี่ยน

- `$"ColumnName"`  
  ใช้อ้างอิง column  
  ต้อง `import spark.implicits._`

- function เช่น `avg`, `count`, `col`  
  ต้อง import `org.apache.spark.sql.functions._`


อ้างอิง
1. https://www.google.com/url?q=https%3A%2F%2Fspark.apache.org%2Fdocs%2Flatest%2Fapi%2Fpython%2Freference%2Findex.html

2.   https://www.flexera.com/blog/finops/apache-spark-with-scala/#spark-and-scala%E2%80%94a-symbiotic-relationship

3.   https://blog.datath.com/cheatsheet-pyspark/?fbclid=IwZXh0bgNhZW0CMTAAAR2TIiC_f7r9Vmo9MCDbOZixubvORxw6qIQ5z0I39fDNy1mBjEXnn-EDvXo_aem_ATAWEUjdxOEBThE9Reem1QUsF8icNmdMsoBoLWrZIpN4hCRZCxV0pfvJz-qeS75PHhXe152qcd-L0qwv2-UnX7Zp#withi_kar_hold_fil_hi_pen_RDD

4. https://www.datacamp.com/community/blog/pyspark-cheat-sheet-python

5. https://www.datacamp.com/community/blog/pyspark-sql-cheat-sheet

6. https://blog.datath.com/cheatsheet-pyspark/?fbclid=IwZXh0bgNhZW0CMTAAAR2TIiC_f7r9Vmo9MCDbOZixubvORxw6qIQ5z0I39fDNy1mBjEXnn-EDvXo_aem_ATAWEUjdxOEBThE9Reem1QUsF8icNmdMsoBoLWrZIpN4hCRZCxV0pfvJz-qeS75PHhXe152qcd-L0qwv2-UnX7Zp#withi_kar_hold_fil_hi_pen_RDD



